# S4-02: 전체 RAG 흐름, BM25, 하이브리드 검색, 리랭킹
**Skilljar L04-L08: Full RAG Flow / Implementing RAG / BM25 / Multi-Index / Reranking**

## 학습 목표
- 전체 RAG 파이프라인 (인덱싱 + 질의)을 구현한다
- BM25 어휘 검색을 구현하고 시맨틱 검색과 비교한다
- 하이브리드 검색 (시맨틱 + BM25 + RRF)을 구현한다
- 리랭킹으로 검색 정밀도를 향상시킨다

## 사전 준비
1. `.env` 파일에 API 키 설정
2. S4_01에서 사용한 문서 데이터와 함수를 재사용합니다

In [ ]:
# 패키지 설치
%pip install anthropic openai python-dotenv chromadb numpy rank-bm25

In [ ]:
# 환경 설정 및 데이터 준비
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from openai import OpenAI
import chromadb
import numpy as np
import re
from rank_bm25 import BM25Okapi

anthropic_client = Anthropic()
openai_client = OpenAI()
chroma_client = chromadb.Client()
model = "claude-sonnet-4-0"

def get_embeddings(texts: list[str]) -> list[list[float]]:
    """여러 텍스트의 임베딩을 한 번에 생성한다."""
    response = openai_client.embeddings.create(
        input=texts, model="text-embedding-3-small"
    )
    return [item.embedding for item in response.data]

def cosine_similarity(vec_a, vec_b) -> float:
    """코사인 유사도 계산"""
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# KDS 문서 데이터
documents = [
    {"id": "kds-4.2.1", "text": "4.2.1 일반 사항: 콘크리트구조 부재의 설계는 극한강도설계법에 따른다. 모든 부재는 소요강도 이상의 설계강도를 가져야 한다. 설계강도 = 강도감소계수(phi) x 공칭강도(Rn).", "metadata": {"clause": "4.2.1", "chapter": "4", "topic": "일반"}},
    {"id": "kds-4.2.2", "text": "4.2.2 강도감소계수: 인장지배 단면의 강도감소계수는 0.85로 한다. 압축지배 단면의 경우 나선철근 부재는 0.70, 기타 부재는 0.65로 한다.", "metadata": {"clause": "4.2.2", "chapter": "4", "topic": "강도감소계수"}},
    {"id": "kds-4.3.1", "text": "4.3.1 축하중을 받는 부재: 축방향 압축력을 받는 부재의 공칭강도 Pn = 0.80 * [0.85 * fck * (Ag - Ast) + fy * Ast]. 여기서 Ag는 전체 단면적, Ast는 철근 단면적이다.", "metadata": {"clause": "4.3.1", "chapter": "4", "topic": "축하중"}},
    {"id": "kds-4.4.1", "text": "4.4.1 최소 철근비: 기둥의 종방향 철근비는 전체 단면적의 1% 이상, 8% 이하로 한다. 최소 4개의 종방향 철근을 배치해야 한다.", "metadata": {"clause": "4.4.1", "chapter": "4", "topic": "철근비"}},
    {"id": "kds-4.4.2", "text": "4.4.2 띠철근 간격: 띠철근 간격은 다음 중 작은 값 이하로 한다. (1) 종방향 철근 지름의 16배 (2) 띠철근 지름의 48배 (3) 기둥 단면의 최소 치수.", "metadata": {"clause": "4.4.2", "chapter": "4", "topic": "띠철근"}},
    {"id": "kds-5.1.1", "text": "5.1.1 보의 휨 설계: 보의 공칭 휨강도 Mn = As * fy * (d - a/2). 여기서 a = As * fy / (0.85 * fck * b). 보의 인장 철근비는 균형 철근비의 75% 이하로 제한한다.", "metadata": {"clause": "5.1.1", "chapter": "5", "topic": "보 휨설계"}},
    {"id": "kds-5.2.1", "text": "5.2.1 보의 전단 설계: 콘크리트가 부담하는 전단강도 Vc = (1/6) * sqrt(fck) * b * d. 전단철근이 부담하는 전단강도 Vs = Av * fy * d / s.", "metadata": {"clause": "5.2.1", "chapter": "5", "topic": "보 전단설계"}},
    {"id": "kds-6.1.1", "text": "6.1.1 내진설계 일반: 내진설계범주 D 이상의 구조물에서 특수 모멘트골조를 사용하는 경우, 기둥의 강도는 보 강도의 1.2배 이상이어야 한다 (강기둥-약보 원칙).", "metadata": {"clause": "6.1.1", "chapter": "6", "topic": "내진설계"}},
]

doc_texts = [d["text"] for d in documents]
print("환경 설정 완료")

---
## Exercise 1: BM25 어휘 검색 구현

BM25 키워드 검색 엔진을 구현하고 시맨틱 검색과 비교하세요.

**요구사항:**
1. `BM25Search` 클래스 구현:
   - `__init__(documents)`: BM25Okapi 인덱스 초기화
   - `_tokenize(text)`: 간단한 공백 + 특수문자 분리 토큰화
   - `search(query, top_k)`: 상위 K개 결과 반환 `[(index, score), ...]`
2. BM25와 시맨틱 검색 결과를 3가지 질의로 비교
   - "KDS 14 20 20 강도감소계수" (전문 코드 포함)
   - "기둥 설계 방법" (일반적 의미)
   - "Pn 공칭강도 계산식" (수식 키워드)

**기대 출력:**
```
질의: KDS 14 20 20 강도감소계수

[BM25 결과]
  1. kds-4.2.2 (점수: 3.xxxx) — 강도감소계수
  2. kds-4.2.1 (점수: 1.xxxx) — 일반
  3. kds-5.1.1 (점수: 0.xxxx) — 보 휨설계

[시맨틱 결과]
  1. kds-4.2.2 (거리: 0.xxxx) — 강도감소계수
  2. kds-4.3.1 (거리: 0.xxxx) — 축하중
  3. kds-4.2.1 (거리: 0.xxxx) — 일반
```

In [ ]:
# TODO: BM25Search 클래스를 구현하고 시맨틱 검색과 비교하세요

queries = [
    "KDS 14 20 20 강도감소계수",
    "기둥 설계 방법",
    "Pn 공칭강도 계산식",
]

In [ ]:
# ===== 정답 =====

class BM25Search:
    """BM25 기반 어휘 검색 엔진"""

    def __init__(self, documents: list[str]):
        self.documents = documents
        self.tokenized = [self._tokenize(doc) for doc in documents]
        self.bm25 = BM25Okapi(self.tokenized)

    def _tokenize(self, text: str) -> list[str]:
        """간단한 토큰화: 공백과 특수문자로 분리"""
        tokens = re.findall(r'\w+', text.lower())
        return tokens

    def search(self, query: str, top_k: int = 3) -> list[tuple[int, float]]:
        """BM25 점수 기반 상위 K개 문서 반환"""
        query_tokens = self._tokenize(query)
        scores = self.bm25.get_scores(query_tokens)
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

# BM25 인덱스 생성
bm25_engine = BM25Search(doc_texts)

# 시맨틱 검색용 ChromaDB
sem_collection = chroma_client.get_or_create_collection(name="sem_compare_ex1")
doc_embeddings = get_embeddings(doc_texts)
sem_collection.add(
    ids=[d["id"] for d in documents],
    documents=doc_texts,
    metadatas=[d["metadata"] for d in documents],
    embeddings=doc_embeddings
)

# 비교 실행
queries = [
    "KDS 14 20 20 강도감소계수",
    "기둥 설계 방법",
    "Pn 공칭강도 계산식",
]

for query in queries:
    print(f"질의: {query}\n")

    # BM25
    bm25_results = bm25_engine.search(query, top_k=3)
    print("[BM25 결과]")
    for idx, score in bm25_results:
        meta = documents[idx]["metadata"]
        print(f"  {documents[idx]['id']} (점수: {score:.4f}) — {meta['topic']}")

    # 시맨틱
    q_emb = get_embeddings([query])[0]
    sem_results = sem_collection.query(
        query_embeddings=[q_emb], n_results=3,
        include=["metadatas", "distances"]
    )
    print("\n[시맨틱 결과]")
    for i in range(len(sem_results["ids"][0])):
        dist = sem_results["distances"][0][i]
        meta = sem_results["metadatas"][0][i]
        print(f"  {sem_results['ids'][0][i]} (거리: {dist:.4f}) — {meta['topic']}")

    print(f"\n{'='*50}\n")

def verify_ex1():
    results = bm25_engine.search("강도감소계수", top_k=1)
    assert documents[results[0][0]]["id"] == "kds-4.2.2", "BM25가 강도감소계수 질의에 kds-4.2.2를 1위로 반환해야 합니다"
    assert results[0][1] > 0, "BM25 점수가 0보다 커야 합니다"
    print("Exercise 1 통과!")

verify_ex1()

---
## Exercise 2: 하이브리드 검색 (RRF) 구현

시맨틱 검색과 BM25 검색을 Reciprocal Rank Fusion으로 결합하는 하이브리드 검색을 구현하세요.

**요구사항:**
1. `reciprocal_rank_fusion(rankings, k=60)` 함수 구현
   - 입력: 여러 랭킹 리스트 `[[(doc_id, score), ...], ...]`
   - 출력: RRF 점수 기준 정렬된 `[(doc_id, rrf_score), ...]`
2. `hybrid_search(query, top_k)` 함수 구현
   - 시맨틱 검색 + BM25 검색 수행
   - RRF로 결합
3. 시맨틱만, BM25만, 하이브리드 결과를 비교

**테스트 질의:**
```python
"KDS 기둥 축하중 공칭강도 Pn 계산"
```

**기대 출력:**
```
질의: KDS 기둥 축하중 공칭강도 Pn 계산

[시맨틱 Top-3] kds-4.3.1, kds-4.2.1, kds-4.4.1
[BM25 Top-3]   kds-4.3.1, kds-4.2.2, kds-4.2.1
[하이브리드 RRF] kds-4.3.1 (0.0xxx), kds-4.2.1 (0.0xxx), ...
```

In [ ]:
# TODO: RRF 함수와 hybrid_search 함수를 구현하세요

test_query = "KDS 기둥 축하중 공칭강도 Pn 계산"

In [ ]:
# ===== 정답 =====

def reciprocal_rank_fusion(
    rankings: list[list[tuple[str, float]]],
    k: int = 60
) -> list[tuple[str, float]]:
    """Reciprocal Rank Fusion으로 여러 검색 결과를 결합한다."""
    rrf_scores = {}
    for ranking in rankings:
        for rank, (doc_id, _score) in enumerate(ranking):
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0.0
            rrf_scores[doc_id] += 1.0 / (k + rank + 1)

    sorted_results = sorted(
        rrf_scores.items(), key=lambda x: x[1], reverse=True
    )
    return sorted_results

def hybrid_search(query: str, top_k: int = 5) -> list[tuple[str, float]]:
    """시맨틱 + BM25 + RRF 하이브리드 검색"""
    # 1. 시맨틱 검색
    q_emb = get_embeddings([query])[0]
    sem = sem_collection.query(
        query_embeddings=[q_emb], n_results=top_k,
        include=["distances"]
    )
    semantic_ranking = [
        (sem["ids"][0][i], 1.0 - sem["distances"][0][i])
        for i in range(len(sem["ids"][0]))
    ]

    # 2. BM25 검색
    bm25_res = bm25_engine.search(query, top_k=top_k)
    bm25_ranking = [
        (documents[idx]["id"], score)
        for idx, score in bm25_res
    ]

    # 3. RRF 결합
    fused = reciprocal_rank_fusion([semantic_ranking, bm25_ranking], k=60)
    return fused[:top_k]

# 비교 실행
test_query = "KDS 기둥 축하중 공칭강도 Pn 계산"
print(f"질의: {test_query}\n")

# 시맨틱만
q_emb = get_embeddings([test_query])[0]
sem = sem_collection.query(query_embeddings=[q_emb], n_results=3, include=["distances"])
print(f"[시맨틱 Top-3] {', '.join(sem['ids'][0])}")

# BM25만
bm25_res = bm25_engine.search(test_query, top_k=3)
bm25_ids = [documents[idx]["id"] for idx, _ in bm25_res]
print(f"[BM25 Top-3]   {', '.join(bm25_ids)}")

# 하이브리드
hybrid_res = hybrid_search(test_query, top_k=5)
print(f"\n[하이브리드 RRF]")
for doc_id, rrf_score in hybrid_res:
    meta = next(d["metadata"] for d in documents if d["id"] == doc_id)
    print(f"  {doc_id} (RRF: {rrf_score:.6f}) — {meta['topic']}")

def verify_ex2():
    # RRF 기본 테스트
    test_rankings = [
        [("A", 0.9), ("B", 0.8), ("C", 0.7)],
        [("B", 0.9), ("A", 0.8), ("D", 0.7)],
    ]
    rrf = reciprocal_rank_fusion(test_rankings, k=60)
    rrf_dict = dict(rrf)
    # A와 B 모두 두 랭킹에 존재하므로 높은 점수
    assert rrf_dict.get("A", 0) > rrf_dict.get("D", 0), "A가 D보다 높아야 합니다"
    assert rrf_dict.get("B", 0) > rrf_dict.get("C", 0), "B가 C보다 높아야 합니다"

    # 하이브리드 검색 테스트
    h_res = hybrid_search("축하중 공칭강도", top_k=3)
    assert len(h_res) >= 3, "하이브리드 검색 결과가 3개 이상이어야 합니다"
    assert h_res[0][0] == "kds-4.3.1", "축하중 질의에 kds-4.3.1이 1위여야 합니다"
    print("\nExercise 2 통과!")

verify_ex2()

---
## Exercise 3: 리랭킹 파이프라인 구현

초기 검색 결과를 리랭킹하는 파이프라인을 구현하세요. 여기서는 Claude를 리랭커로 활용합니다 (실습용, 프로덕션에서는 Cohere Rerank 등 전용 모델 사용).

**요구사항:**
1. `claude_rerank(query, documents, top_n)` 함수 구현
   - Claude에게 질의와 문서 목록을 제공하여 관련성 순위를 매기도록 요청
   - JSON 형식으로 결과 파싱
2. 초기 검색 → 리랭킹 → 최종 Top-K 파이프라인 구성
3. 리랭킹 전후 결과 비교

**테스트 질의:**
```python
"내진 설계에서 기둥과 보의 강도 관계"
```

**기대 출력:**
```
[리랭킹 전 Top-5]
  1. kds-6.1.1 — 내진설계
  2. kds-4.3.1 — 축하중
  ...

[리랭킹 후 Top-3]
  1. kds-6.1.1 (점수: 0.95) — 내진설계: 직접 관련
  2. kds-5.1.1 (점수: 0.60) — 보 휨설계: 보 강도 관련
  3. kds-4.3.1 (점수: 0.55) — 축하중: 기둥 강도 관련
```

In [ ]:
# TODO: claude_rerank 함수와 리랭킹 파이프라인을 구현하세요

rerank_query = "내진 설계에서 기둥과 보의 강도 관계"

In [ ]:
# ===== 정답 =====
import json

def claude_rerank(query: str, docs: list[dict], top_n: int = 3) -> list[dict]:
    """Claude를 사용한 리랭킹 (실습용).

    각 문서의 질의에 대한 관련성 점수(0-1)를 Claude에게 요청한다.
    """
    doc_list = "\n".join(
        f"[문서 {i}] (ID: {d['id']}) {d['text'][:100]}"
        for i, d in enumerate(docs)
    )

    prompt = f"""당신은 문서 관련성 평가 전문가입니다.

질의: {query}

다음 문서들의 질의에 대한 관련성 점수(0.0~1.0)를 평가하세요.

{doc_list}

JSON 배열로 답변하세요. 관련성 점수가 높은 순으로 정렬하세요.
형식: [{{"doc_index": 0, "id": "...", "score": 0.95, "reason": "..."}}]"""

    messages = [{"role": "user", "content": prompt}]
    messages.append({"role": "assistant", "content": "```json\n"})

    response = anthropic_client.messages.create(
        model=model,
        max_tokens=1024,
        temperature=0.0,
        messages=messages,
        stop_sequences=["```"]
    )

    result = json.loads(response.content[0].text.strip())
    # 점수 순 정렬 후 top_n 반환
    result.sort(key=lambda x: x["score"], reverse=True)
    return result[:top_n]

# 파이프라인 실행
rerank_query = "내진 설계에서 기둥과 보의 강도 관계"
print(f"질의: {rerank_query}\n")

# 1. 초기 하이브리드 검색 (Top-5)
initial = hybrid_search(rerank_query, top_k=5)
doc_map = {d["id"]: d for d in documents}
initial_docs = [doc_map[doc_id] for doc_id, _ in initial if doc_id in doc_map]

print("[리랭킹 전 Top-5]")
for i, (doc_id, rrf) in enumerate(initial):
    meta = doc_map[doc_id]["metadata"]
    print(f"  {i+1}. {doc_id} (RRF: {rrf:.6f}) — {meta['topic']}")

# 2. Claude 리랭킹 (Top-3)
reranked = claude_rerank(rerank_query, initial_docs, top_n=3)

print(f"\n[리랭킹 후 Top-3]")
for r in reranked:
    doc_id = r["id"]
    score = r["score"]
    reason = r.get("reason", "")
    meta = doc_map[doc_id]["metadata"]
    print(f"  {doc_id} (점수: {score:.2f}) — {meta['topic']}: {reason[:40]}")

def verify_ex3():
    assert len(reranked) == 3, "리랭킹 결과가 3개여야 합니다"
    assert all("score" in r and "id" in r for r in reranked), "각 결과에 score와 id가 있어야 합니다"
    assert reranked[0]["score"] >= reranked[1]["score"], "점수 순 정렬이어야 합니다"
    # 내진설계 문서가 상위에 있어야 함
    top_ids = [r["id"] for r in reranked]
    assert "kds-6.1.1" in top_ids, "kds-6.1.1이 Top-3에 포함되어야 합니다"
    print("\nExercise 3 통과!")

verify_ex3()

---
## 건축공학 실습: 전체 하이브리드 RAG + 리랭킹 파이프라인

시맨틱 검색, BM25, RRF 융합, 리랭킹을 모두 결합한 **종합 RAG 파이프라인**을 구현하세요. 검색 결과를 Claude에게 전달하여 KDS 기준에 근거한 답변을 생성합니다.

**요구사항:**
1. `FullRAGPipeline` 클래스 구현:
   - `__init__(documents)`: 시맨틱 인덱스 + BM25 인덱스 초기화
   - `search(query, top_k)`: 하이브리드 검색 (시맨틱 + BM25 + RRF)
   - `generate(question, top_k)`: 검색 → 프롬프트 조립 → Claude 응답 생성
2. 3가지 건축공학 질의로 테스트
3. 각 답변에 KDS 조항 번호가 인용되는지 확인

**테스트 질의:**
```python
"500x500 기둥, fck=24MPa, 8-D25 배치 시 축하중 강도를 계산하세요."
"보의 전단 설계에서 콘크리트 전단강도 Vc를 구하는 공식은?"
"내진설계범주 D에서 기둥-보 접합부의 설계 원칙은?"
```

In [ ]:
# TODO: FullRAGPipeline 클래스를 구현하세요

test_queries = [
    "500x500 기둥, fck=24MPa, 8-D25 배치 시 축하중 강도를 계산하세요.",
    "보의 전단 설계에서 콘크리트 전단강도 Vc를 구하는 공식은?",
    "내진설계범주 D에서 기둥-보 접합부의 설계 원칙은?",
]

In [ ]:
# ===== 정답 =====

class FullRAGPipeline:
    """시맨틱 + BM25 + RRF + Claude 종합 RAG 파이프라인"""

    def __init__(self, documents: list[dict]):
        self.documents = documents
        self.doc_texts = [d["text"] for d in documents]
        self.doc_map = {d["id"]: d for d in documents}

        # 시맨틱 인덱스
        self.collection = chroma_client.get_or_create_collection(
            name="full_rag_pipeline"
        )
        embeddings = get_embeddings(self.doc_texts)
        self.collection.add(
            ids=[d["id"] for d in documents],
            documents=self.doc_texts,
            metadatas=[d["metadata"] for d in documents],
            embeddings=embeddings
        )

        # BM25 인덱스
        self.bm25 = BM25Search(self.doc_texts)

        print(f"파이프라인 초기화 완료: {len(documents)}개 문서")

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        """하이브리드 검색 (시맨틱 + BM25 + RRF)"""
        # 시맨틱
        q_emb = get_embeddings([query])[0]
        sem = self.collection.query(
            query_embeddings=[q_emb], n_results=top_k,
            include=["distances"]
        )
        sem_ranking = [
            (sem["ids"][0][i], 1.0 - sem["distances"][0][i])
            for i in range(len(sem["ids"][0]))
        ]

        # BM25
        bm25_res = self.bm25.search(query, top_k=top_k)
        bm25_ranking = [
            (self.documents[idx]["id"], score) for idx, score in bm25_res
        ]

        # RRF
        fused = reciprocal_rank_fusion([sem_ranking, bm25_ranking], k=60)

        results = []
        for doc_id, rrf_score in fused[:top_k]:
            if doc_id in self.doc_map:
                results.append({
                    **self.doc_map[doc_id],
                    "rrf_score": rrf_score
                })
        return results

    def generate(self, question: str, top_k: int = 3) -> str:
        """검색 → 프롬프트 조립 → Claude 응답 생성"""
        results = self.search(question, top_k)

        context = "\n\n".join(
            f"[KDS {r['metadata']['clause']}] {r['text']}"
            for r in results
        )

        prompt = f"""당신은 KDS 콘크리트구조 설계기준 전문가입니다.

검색된 KDS 조항:
{context}

규칙:
1. 반드시 위 조항을 근거로 답변하세요
2. 조항 번호를 인용하세요 (예: KDS 4.3.1에 따르면...)
3. 계산이 필요하면 단계별로 보여주세요
4. 검색 결과에 없는 내용은 "해당 조항이 검색되지 않았습니다"라고 명시하세요

질문: {question}"""

        response = anthropic_client.messages.create(
            model=model,
            max_tokens=1500,
            temperature=0.1,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

# 파이프라인 실행
pipeline = FullRAGPipeline(documents)

test_queries = [
    "500x500 기둥, fck=24MPa, 8-D25 배치 시 축하중 강도를 계산하세요.",
    "보의 전단 설계에서 콘크리트 전단강도 Vc를 구하는 공식은?",
    "내진설계범주 D에서 기둥-보 접합부의 설계 원칙은?",
]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    answer = pipeline.generate(q)
    print(f"\nA: {answer}")

def verify_structural():
    # 축하중 계산 질의
    answer = pipeline.generate("기둥 축하중 강도 Pn 계산", top_k=2)
    assert "4.3.1" in answer or "Pn" in answer, "축하중 질의에 4.3.1 또는 Pn이 포함되어야 합니다"

    # 전단 설계 질의
    answer2 = pipeline.generate("보 전단강도 Vc", top_k=2)
    assert "5.2.1" in answer2 or "Vc" in answer2, "전단 질의에 5.2.1 또는 Vc가 포함되어야 합니다"

    print("\n건축공학 실습 통과!")

verify_structural()